# Setup: Postgres `bench_patch_random_read` Table

## Schema Description

This notebook creates and seeds the `bench_patch_random_read` unlogged table used for the
**Random read of 1000 patches (Postgres)** benchmark.

The schema mirrors the `patch` table from `db_technical_design.md`:

| Column       | Type         | Description                                          |
|-------------|--------------|------------------------------------------------------|
| id          | BIGSERIAL PK | Sequential primary key (matches patch_id concept)    |
| patch_uid   | INT          | Unique patch identifier                              |
| gt_label    | INT          | Ground truth label (0–9, random)                     |
| event_ts    | TIMESTAMPTZ  | Timestamp when ground truth was set                  |
| image_id    | INT          | Foreign-key-like reference to source image (1–100)   |
| working_mag | FLOAT        | Working magnification level (1.0–4.0, random)        |

**Table type**: `UNLOGGED` — avoids WAL overhead during seeding; acceptable for a benchmark table.

**Index**: `PRIMARY KEY` B-tree index on `id` (matching production `patch` table index).

**Row count**: 1,000,000 (1M rows)

**Seeding method**: Server-side `generate_series` in 100k-row batches for fast population.

## Connection
Reads from environment variables `DB_HOST`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`;
falls back to the prototyping defaults if not set.

In [ ]:
import os
import time
import psycopg2

# ---------------------------------------------------------------------------
# Connection parameters — read from env vars, fall back to prototyping defaults
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')

DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE     = 'bench_patch_random_read'
TOTAL     = 1_000_000
BATCH     = 100_000

conn = psycopg2.connect(DSN)
conn.autocommit = False
cur = conn.cursor()

# Report PG version
cur.execute('SELECT version();')
pg_ver = cur.fetchone()[0].split(',')[0]
print(f'Connected to {pg_ver}')

# ---------------------------------------------------------------------------
# Drop + recreate table
# ---------------------------------------------------------------------------
cur.execute(f'DROP TABLE IF EXISTS {TABLE};')
conn.commit()
print(f'Dropped existing {TABLE} (if present).')

cur.execute(f'''
    CREATE UNLOGGED TABLE {TABLE} (
        id          BIGSERIAL PRIMARY KEY,
        patch_uid   INT,
        gt_label    INT,
        event_ts    TIMESTAMPTZ,
        image_id    INT,
        working_mag FLOAT
    );
''')
conn.commit()
print(f'Table {TABLE} created.')

# ---------------------------------------------------------------------------
# Seed 1M rows via server-side generate_series (fast)
# ---------------------------------------------------------------------------
t0 = time.time()
for start in range(1, TOTAL + 1, BATCH):
    end = min(start + BATCH - 1, TOTAL)
    cur.execute(f'''
        INSERT INTO {TABLE} (patch_uid, gt_label, event_ts, image_id, working_mag)
        SELECT
            gs,
            (random() * 9)::INT,
            NOW() - (random() * INTERVAL '365 days'),
            (random() * 99 + 1)::INT,
            (random() * 3 + 1)
        FROM generate_series({start}, {end}) AS gs;
    ''')
    conn.commit()
    print(f'  Seeded rows {start} \u2013 {end}')

elapsed = time.time() - t0
print(f'Seeding complete in {elapsed:.1f}s')

# ---------------------------------------------------------------------------
# Verify
# ---------------------------------------------------------------------------
cur.execute(f'SELECT COUNT(*) FROM {TABLE};')
count = cur.fetchone()[0]
print(f'Final row count: {count:,}')
assert count == TOTAL, f'Expected {TOTAL} rows, got {count}'

conn.close()

Connected to PostgreSQL 15.17
Dropped existing bench_patch_random_read (if present).
Table bench_patch_random_read created.
  Seeded rows 1 – 100000
  Seeded rows 100001 – 200000
  Seeded rows 200001 – 300000
  Seeded rows 300001 – 400000
  Seeded rows 400001 – 500000
  Seeded rows 500001 – 600000
  Seeded rows 600001 – 700000
  Seeded rows 700001 – 800000
  Seeded rows 800001 – 900000
  Seeded rows 900001 – 1000000
Seeding complete in 1.8s
Final row count: 1,000,000


In [ ]:
# ---------------------------------------------------------------------------
# TEARDOWN  — run this cell to clean up after benchmarking
# ---------------------------------------------------------------------------
import os
import psycopg2

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

conn = psycopg2.connect(DSN)
conn.autocommit = True
cur = conn.cursor()
cur.execute('DROP TABLE IF EXISTS bench_patch_random_read;')
conn.close()
print('Teardown: bench_patch_random_read dropped.')

Teardown: bench_patch_random_read dropped.
